# schedule_dashboard_notify_v1

目标：验证 dashboard refresh 的通知可以在默认兼容模式和显式详细模式之间切换。详细通知用于把 top model、manifest 来源、analysis preflight 状态和 export 路径推给用户。

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists() and (path / 'src').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.schedule import ModelDashboardPipelineConfig, run_model_dashboard_refresh_once

CARD = REPO_ROOT / 'research/schedule_dashboard_notify_v1'
LOCKS = CARD / 'locks'
CARD.mkdir(parents=True, exist_ok=True)
{'card': str(CARD.relative_to(REPO_ROOT))}

{'card': 'research/schedule_dashboard_notify_v1'}

## Matrix runner

这里使用最小旧 sidecar 产物，打开 auto backfill 后，schedule 会在 dashboard export 前生成 `analysis_manifest.json`。

In [2]:
def legacy_matrix_runner(*, card_dir: Path, start: str, end: str, train_end: str, model_names, top_n_factors: int):
    card_dir.mkdir(parents=True, exist_ok=True)
    (card_dir / 'analysis_summary.json').write_text(json.dumps({
        'schema_version': 'analysis.report.v1',
        'status': 'legacy_matrix_completed',
        'start': start,
        'end': end,
        'train_end': train_end,
        'model_count': len(tuple(model_names)),
    }, ensure_ascii=False, indent=2), encoding='utf-8')
    pd.DataFrame([
        {'rank': 1, 'model': 'ridge_mse', 'return_pct': 2.7, 'relative_return_pct': 1.1, 'stability_score': 2.0},
        {'rank': 2, 'model': 'gcn_mse', 'return_pct': 1.1, 'relative_return_pct': -0.4, 'stability_score': 0.5},
    ]).to_csv(card_dir / 'model_comparison.csv', index=False)
    pd.DataFrame([{'model': 'ridge_mse', 'regime': 'bull', 'return_pct': 2.7}]).to_csv(card_dir / 'regime_metrics.csv', index=False)
    pd.DataFrame([{'model': 'ridge_mse', 'window': 'test_2026', 'return_pct': 2.7}]).to_csv(card_dir / 'window_metrics.csv', index=False)
    return {'status': 'feature_set_matrix_completed', 'variant_count': len(tuple(model_names))}


## 默认通知保持兼容

不开 `notify_details` 时，通知仍然是旧格式 `job:success`，避免破坏已有自动化。

In [3]:
default_events = []
default_config = ModelDashboardPipelineConfig(
    name='dashboard_notify_default',
    card_dir=CARD / 'default_card',
    dashboard_output_path=CARD / 'default_dashboard_export.json',
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'gcn_mse'),
    analysis_manifest_preflight=True,
    analysis_manifest_auto_backfill=True,
    analysis_manifest_dataset_name='schedule_notify_default',
    analysis_manifest_universe=('SOXL',),
    analysis_manifest_calendar_path='data/qlib/soxl/calendars/day.txt',
)
default_result = run_model_dashboard_refresh_once(
    default_config,
    lock_dir=LOCKS,
    notify=default_events.append,
    matrix_runner=legacy_matrix_runner,
)
assert default_result.status == 'success'
assert default_events == ['dashboard_notify_default:success']
default_events

['dashboard_notify_default:success']

## 详细通知

开启 `notify_details=True` 后，通知包含 dashboard export 关键上下文。

In [4]:
detail_events = []
detail_config = ModelDashboardPipelineConfig(
    name='dashboard_notify_detail',
    card_dir=CARD / 'detail_card',
    dashboard_output_path=CARD / 'detail_dashboard_export.json',
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'gcn_mse'),
    analysis_manifest_preflight=True,
    analysis_manifest_auto_backfill=True,
    analysis_manifest_dataset_name='schedule_notify_detail',
    analysis_manifest_universe=('SOXL',),
    analysis_manifest_calendar_path='data/qlib/soxl/calendars/day.txt',
    notify_details=True,
)
detail_result = run_model_dashboard_refresh_once(
    detail_config,
    lock_dir=LOCKS,
    notify=detail_events.append,
    matrix_runner=legacy_matrix_runner,
)
assert detail_result.status == 'success'
assert 'top_model=ridge_mse' in detail_events[0]
assert 'dashboard_manifest=analysis_manifest' in detail_events[0]
assert 'analysis_preflight=backfilled' in detail_events[0]
assert 'attempts=1' in detail_events[0]
detail_events

['dashboard_notify_detail:success | top_model=ridge_mse | dashboard_manifest=analysis_manifest | analysis_preflight=backfilled | attempts=1 | export=/Users/rziiiii/dev/workspace/research/schedule_dashboard_notify_v1/detail_dashboard_export.json']

In [5]:
summary = {
    'schema_version': 'research.schedule_dashboard_notify_v1',
    'status': 'completed',
    'default_notification': default_events[0],
    'detailed_notification': detail_events[0],
    'detail_top_model': detail_result.payload['top_model'],
    'detail_dashboard_manifest_kind': detail_result.payload['dashboard_manifest_kind'],
    'detail_analysis_preflight_status': detail_result.payload['analysis_manifest_preflight_status'],
    'detail_attempts': detail_result.payload['attempts'],
    'conclusion': 'schedule dashboard refresh 可选择发送包含模型、manifest、preflight 和 export 的详细通知。',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
(CARD / 'notification_events.json').write_text(json.dumps({'default': default_events, 'detail': detail_events}, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'research.schedule_dashboard_notify_v1',
 'status': 'completed',
 'default_notification': 'dashboard_notify_default:success',
 'detailed_notification': 'dashboard_notify_detail:success | top_model=ridge_mse | dashboard_manifest=analysis_manifest | analysis_preflight=backfilled | attempts=1 | export=/Users/rziiiii/dev/workspace/research/schedule_dashboard_notify_v1/detail_dashboard_export.json',
 'detail_top_model': 'ridge_mse',
 'detail_dashboard_manifest_kind': 'analysis_manifest',
 'detail_analysis_preflight_status': 'backfilled',
 'detail_attempts': 1,
 'conclusion': 'schedule dashboard refresh 可选择发送包含模型、manifest、preflight 和 export 的详细通知。'}

## 中文结论

- 做了什么：为 dashboard refresh 增加显式详细通知模式，并验证默认通知保持兼容。
- 核心指标：默认通知仍是 `job:success`；详细通知包含 `top_model`、`dashboard_manifest`、`analysis_preflight`、`attempts`、`export`。
- regime 差异：本轮不新增 regime 计算；详细通知把 dashboard manifest 状态带出去，便于用户知道 regime/window 展示是否来自 manifest-first。
- 是否晋升：这是 `src/schedule` 框架能力，可保留；不是策略晋升。
- 风险：详细通知只在 `notify_details=True` 时启用，避免改变已有自动化的消息解析。
- 下一步：可以把通知模板扩展到数据刷新、模型刷新和 paper API replay。